<a href="https://colab.research.google.com/github/lemwaizz/formative2_G35/blob/main/Formative_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Loading Data

In [11]:
from google.colab import drive
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.fft import fft, fftfreq
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import warnings
warnings.filterwarnings('ignore')

# Set plot style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Mount Google Drive
drive.mount('/content/drive')

# Set path to the data folders
BASE_PATH = '/content/drive/MyDrive/hmm/'
MWAI_PATH = os.path.join(BASE_PATH, 'mwai_data')
WENGS_PATH = os.path.join(BASE_PATH, 'wengel_data')

print(f"\nData directories:")
print(f"  Dennis: {MWAI_PATH}")
print(f"  Wengel: {WENGS_PATH}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Data directories:
  Dennis: /content/drive/MyDrive/hmm/mwai_data
  Wengel: /content/drive/MyDrive/hmm/wengel_data


## Data Merging

In [16]:
# Load accelerometer and gyroscope data from a single recording folder, which will return merged dataframe with both sensors synchronized.
def load_single_recording(recording_path):
    try:
        acc_path = os.path.join(recording_path, 'Accelerometer.csv')
        gyro_path = os.path.join(recording_path, 'Gyroscope.csv')

        if not os.path.exists(acc_path) or not os.path.exists(gyro_path):
            print(f"    Missing Accelerometer.csv or Gyroscope.csv in {recording_path}")
            return None

        acc_df = pd.read_csv(acc_path)
        gyro_df = pd.read_csv(gyro_path)

        if acc_df.empty or gyro_df.empty:
            print(f"    Accelerometer.csv or Gyroscope.csv is empty in {recording_path}")
            return None

        # Handle different time column formats
        if 'seconds_elapsed' not in acc_df.columns and 'time' in acc_df.columns:
            acc_df['seconds_elapsed'] = (acc_df['time'] - acc_df['time'].iloc[0]) / 1e9
        if 'seconds_elapsed' not in gyro_df.columns and 'time' in gyro_df.columns:
            gyro_df['seconds_elapsed'] = (gyro_df['time'] - gyro_df['time'].iloc[0]) / 1e9

        # Ensure 'seconds_elapsed' exists for merging
        if 'seconds_elapsed' not in acc_df.columns or 'seconds_elapsed' not in gyro_df.columns:
            print(f"    Missing 'seconds_elapsed' column after processing time for {recording_path}")
            return None

        # Rename columns for clarity
        acc_df = acc_df.rename(columns = {'x': 'x_acc', 'y': 'y_acc', 'z': 'z_acc'})
        gyro_df = gyro_df.rename(columns = {'x': 'x_gyro', 'y': 'y_gyro', 'z': 'z_gyro'})

        # Merge accelerometer and gyroscope data
        merged = pd.merge_asof(
            acc_df.sort_values('seconds_elapsed')[['seconds_elapsed', 'x_acc', 'y_acc', 'z_acc']],
            gyro_df.sort_values('seconds_elapsed')[['seconds_elapsed', 'x_gyro', 'y_gyro', 'z_gyro']],
            on='seconds_elapsed',
            direction='nearest'
        )
        print(f"    Successfully merged {len(merged)} samples from {recording_path}")
        return merged

    except Exception as e:
        print(f"    Error in load_single_recording for {recording_path}: {str(e)}")
        return None

# Load all recordings for a specific activity and participant
def load_activity_data(base_path, activity, participant_name):
    activity_path = os.path.join(base_path, activity)

    if not os.path.exists(activity_path):
        print(f"Activity folder not found: {activity_path}")
        return None

    all_recordings = []
    print(f"\nLoading {activity} for {participant_name}:")

    merged_data = load_single_recording(activity_path)

    if merged_data is not None and len(merged_data) > 0:
        merged_data['participant'] = participant_name
        merged_data['activity'] = activity
        merged_data['recording_id'] = f"{participant_name}_{activity}_recording_01"
        all_recordings.append(merged_data)
        print(f"  {activity} data appended: {len(merged_data)} samples")
    else:
        print(f"  No valid sensor data found directly in {activity_path} or merged data was empty.")

    if all_recordings:
        combined = pd.concat(all_recordings, ignore_index = True)
        print(f"Total: {len(combined)} samples from {len(all_recordings)} recordings for {activity}")
        return combined
    return None

## Loading Data from both participants

### Checking files in MWAI_PATH (Dennis's data)

In [13]:
print(f"Contents of {MWAI_PATH}:")
for item in os.listdir(MWAI_PATH):
    item_path = os.path.join(MWAI_PATH, item)
    if os.path.isdir(item_path):
        print(f"  Directory: {item}/")
        for sub_item in os.listdir(item_path):
            sub_item_path = os.path.join(item_path, sub_item)
            if os.path.isdir(sub_item_path):
                print(f"    Sub-directory: {sub_item}/")
                for file in os.listdir(sub_item_path):
                    print(f"      File: {file}")
            else:
                print(f"    File: {sub_item}")
    else:
        print(f"  File: {item}")


Contents of /content/drive/MyDrive/hmm/mwai_data:
  Directory: jumping/
    File: Accelerometer.csv
    File: Gyroscope.csv
    File: Metadata.csv
    File: AccelerometerUncalibrated.csv
    File: GyroscopeUncalibrated.csv
    File: Annotation.csv
  Directory: walking/
    File: Annotation.csv
    File: Metadata.csv
    File: Gyroscope.csv
    File: AccelerometerUncalibrated.csv
    File: Accelerometer.csv
    File: GyroscopeUncalibrated.csv
  Directory: mwai_unextracted/
    File: Dennis_Jumping_3-2026-03-02_12-59-30.zip
    File: Dennis_Walking_5-2026-03-02_12-47-09.zip
    File: Dennis_Standing_6-2026-03-02_12-32-25.zip
    File: Dennis_Walking_3-2026-03-02_12-43-49.zip
    File: Still_3-2026-03-02_13-11-45.zip
    File: Dennis_Walking_4-2026-03-02_12-44-29.zip
    File: Still_1-2026-03-02_13-11-20.zip
    File: Still_5-2026-03-02_13-12-11.zip
    File: Dennis_Walking_2-2026-03-02_12-43-12.zip
    File: Dennis_Walking_1-2026-03-02_12-42-30.zip
    File: Still_4-2026-03-02_13-11-58.z

### Checking files in WENGS_PATH (Wengs's data)

In [14]:
print(f"Contents of {WENGS_PATH}:")
for item in os.listdir(WENGS_PATH):
    item_path = os.path.join(WENGS_PATH, item)
    if os.path.isdir(item_path):
        print(f"  Directory: {item}/")
        for sub_item in os.listdir(item_path):
            sub_item_path = os.path.join(item_path, sub_item)
            if os.path.isdir(sub_item_path):
                print(f"    Sub-directory: {sub_item}/")
                for file in os.listdir(sub_item_path):
                    print(f"      File: {file}")
            else:
                print(f"    File: {sub_item}")
    else:
        print(f"  File: {item}")


Contents of /content/drive/MyDrive/hmm/wengel_data:
  Directory: standing/
    File: AccelerometerUncalibrated.csv
    File: Accelerometer.csv
    File: Annotation.csv
    File: Gyroscope.csv
    File: GyroscopeUncalibrated.csv
    File: Metadata.csv
  Directory: jumping/
    File: Metadata.csv
    File: AccelerometerUncalibrated.csv
    File: Gyroscope.csv
    File: GyroscopeUncalibrated.csv
    File: Accelerometer.csv
    File: Annotation.csv
  Directory: walking/
    File: GyroscopeUncalibrated.csv
    File: Annotation.csv
    File: AccelerometerUncalibrated.csv
    File: Metadata.csv
    File: Gyroscope.csv
    File: Accelerometer.csv
  Directory: wengs_unextracted/
    File: w_walking_6_-2026-03-02_12-55-20.zip
    File: w_standing_2_-2026-03-02_12-33-52.zip
    File: w_walking_1_-2026-03-02_12-51-50.zip
    File: Still_12-2026-03-02_13-16-01.zip
    File: w_standing_5_-2026-03-02_12-37-21.zip
    File: w_jumping_2_-2026-03-02_13-06-00.zip
    File: w_walking_5_-2026-03-02_12-54-4

In [18]:
activities = ['walking', 'standing', 'jumping', 'still']
all_data = []

# Load Dennis' data
print("\nDennis' Data:")
for activity in activities:
    data = load_activity_data(MWAI_PATH, activity, 'Dennis')
    if data is not None:
        all_data.append(data)

# Load Marion's data
print("\nWengs' Data:")
for activity in activities:
    data = load_activity_data(WENGS_PATH, activity, 'Wengs')
    if data is not None:
        all_data.append(data)

# Combine all data
complete_dataset = pd.concat(all_data, ignore_index = True)

print(f"\nTotal samples collected: {len(complete_dataset):,}")
print(f"\nSamples per activity:")
for activity, count in complete_dataset.groupby('activity').size().items():
    print(f"  {activity.capitalize():<12}: {count:>6,} samples")
print(f"\nTotal recordings: {complete_dataset['recording_id'].nunique()}")


Dennis' Data:

Loading walking for Dennis:
    Successfully merged 1096 samples from /content/drive/MyDrive/hmm/mwai_data/walking
  walking data appended: 1096 samples
Total: 1096 samples from 1 recordings for walking

Loading standing for Dennis:
    Successfully merged 1071 samples from /content/drive/MyDrive/hmm/mwai_data/standing
  standing data appended: 1071 samples
Total: 1071 samples from 1 recordings for standing

Loading jumping for Dennis:
    Successfully merged 1095 samples from /content/drive/MyDrive/hmm/mwai_data/jumping
  jumping data appended: 1095 samples
Total: 1095 samples from 1 recordings for jumping

Loading still for Dennis:
    Successfully merged 1107 samples from /content/drive/MyDrive/hmm/mwai_data/still
  still data appended: 1107 samples
Total: 1107 samples from 1 recordings for still

Wengs' Data:

Loading walking for Wengs:
    Successfully merged 1071 samples from /content/drive/MyDrive/hmm/wengel_data/walking
  walking data appended: 1071 samples
Tota